<div style="font-size:12px">

## DynaWatch Calibrator – Ground Truth & Waveform Engine ##

This notebook implements the mathematical core of the DynaWatch PMU calibrator.

It generates:

1. **Ground Truth reports** at 100 fps  
2. **Seven-channel waveform datasets** at ~8 kS/s with drift and jitter  

The implementation follows the definitions of **IEC/IEEE 60255-118-1**.

---

### Architecture Overview

**Input:**  
Master file (JSON) for a specific performance class (P or M) and a selected test ID.

**Output:**  
Matched pairs of:
- Ground truth reports (JSON)
- Waveform datasets (JSON)

for each atomic test case defined in the master file.

---

### Execution Flow

The system is orchestrated by:

**TestCampaignController** (main orchestrator)

which coordinates the following layers:

→ **Atomic Test Expansion**  
Reads the master file and expands it into individual atomic test cases.  
Each atomic test varies exactly one parameter in accordance with IEC/IEEE 60255-118-1.

→ **TimelineBuilder**  
Constructs a continuous time structure defining:
- Warm-up intervals  
- Event start times  
- Event durations  
- Transition behaviour  

This ensures strict temporal alignment with absolute time.

→ **Physics States**  
Defines the electrical state at every instant:
- Three-phase voltages  
- Three-phase currents  
- Neutral current  
- Harmonics (with correct sequence)  
- OOB interference  
- Modulation or ramp dynamics  

All quantities are analytically defined functions of time.

→ **GroundTruthGenerator**  
Evaluates the analytical definitions at each reporting instant (100 fps) to produce:
- Synchrophasor phase  
- Frequency  
- ROCOF  
- RMS magnitudes per phase  

Harmonics and OOB components are excluded from ground truth by design.

→ **WaveformGenerator**  
Synthesizes the seven physical channels:
- Va, Vb, Vc  
- Ia, Ib, Ic  
- In  

Includes:
- Continuous phase evolution
- Harmonic sequence enforcement  
- Optional OOB injection
- Per-phase power factor
- Optional asymmetry  
- Drifted sampling rate  
- Gaussian jitter  
- ADC-safe clipping  
- Sampling timestamps are stored in nanoseconds relative to the absolute start time.

---

### Design Principles

- Strict separation of responsibilities between layers  
- Analytical derivatives only (no numerical differentiation)  
- Deterministic phase continuity across segments  
- One-parameter-at-a-time atomic testing  
- Hardware-aware waveform bounds  

This layered architecture guarantees mathematical correctness, reproducibility, and compliance alignment.


</div>

<div style="font-size:12px">

### Core Time & Physics Infrastructure

The following classes form the analytical backbone of the test engine.

They do not generate output directly.  
Instead, they define how electrical quantities evolve continuously in time.

- **SteadyStatePhysics**  
  Defines analytical expressions for θ(t), its derivatives, and RMS magnitudes  
  for steady operating conditions.

- **Segment**  
  Represents a time-bounded interval with a specific physics state.  
  Ensures phase continuity by carrying forward θ offsets between segments.

- **Timeline**  
  Organizes ordered segments into a continuous absolute time structure.  
  Provides time-to-segment resolution and local time mapping.

These classes guarantee:
- Analytical phase continuity
- Deterministic time evolution
- Clean separation between time modelling and signal generation
</div>

In [9]:
from numpy import pi


class SteadyStatePhysics:

    def __init__(self, f, master, atomic, nominal=False):

        self.f = f
        self.master = master
        self.atomic = atomic
        self.nominal = nominal

        # --------------------------------------------
        # Project-level parameters
        # --------------------------------------------

        self.f0 = master["project"]["f0"]

        # --------------------------------------------
        # Scaling
        # --------------------------------------------

        self.Vrms_nominal = master["scaling"]["voltage"]["Vrms_nominal"]
        self.Irms_nominal = master["scaling"]["current"]["Irms_nominal"]

        # --------------------------------------------
        # Duration
        # --------------------------------------------

        self._duration = master["steady_state"]["duration_sec"]

        # --------------------------------------------
        # Power Factor (degrees → radians)
        # --------------------------------------------

        pf_deg = master["steady_state"].get(
            "power_factor_deg",
            {"Ia": 0.0, "Ib": 0.0, "Ic": 0.0}
        )

        self.pf = {
            "Ia": pf_deg["Ia"] * pi / 180.0,
            "Ib": pf_deg["Ib"] * pi / 180.0,
            "Ic": pf_deg["Ic"] * pi / 180.0
        }

        # --------------------------------------------
        # Harmonics
        # --------------------------------------------

        params = atomic.get("parameters", {})
        self.harmonics = [params["harmonic"]] if "harmonic" in params else []

        # --------------------------------------------
        # Out-of-band components
        # --------------------------------------------

        self.oob = [params["oob"]] if "oob" in params else []

        # --------------------------------------------
        # Per-test amplitude scaling (if atomic defines PU)
        # --------------------------------------------

        params = atomic.get("parameters", {})

        self.voltage_pu = params.get("voltage_scale", 1.0)
        self.current_pu = params.get("current_scale", 1.0)

    # ==================================================
    # Phase evolution
    # ==================================================

    def theta(self, t):
        return 2 * pi * self.f * t

    def dtheta_dt(self, t):
        return 2 * pi * self.f

    def d2theta_dt2(self, t):
        return 0.0

    # ==================================================
    # Duration
    # ==================================================

    def duration(self):
        return self._duration

    # ==================================================
    # RMS per phase
    # ==================================================

    def voltage_rms_per_phase(self, t):

        Vrms = self.Vrms_nominal * self.voltage_pu

        return {
            "Va": Vrms,
            "Vb": Vrms,
            "Vc": Vrms
        }

    def current_rms_per_phase(self, t):

        Irms = self.Irms_nominal * self.current_pu

        return {
            "Ia": Irms,
            "Ib": Irms,
            "Ic": Irms
        }

    # ==================================================
    # Power factor per phase
    # ==================================================

    def power_factor_per_phase(self, t):
        return self.pf


class RampPhysics:

    def __init__(self, master, atomic, f_start, ramp_rate):

        self.master = master
        self.atomic = atomic
        self.f_start = f_start
        self.ramp_rate = ramp_rate

        # --------------------------------------------
        # Project-level parameters
        # --------------------------------------------

        self.f0 = master["project"]["f0"]

        # --------------------------------------------
        # Scaling
        # --------------------------------------------

        self.Vrms_nominal = master["scaling"]["voltage"]["Vrms_nominal"]
        self.Irms_nominal = master["scaling"]["current"]["Irms_nominal"]

        # --------------------------------------------
        # Duration
        # --------------------------------------------
        
        params = atomic.get("parameters", {})
        # Safe default using f_start if f_end missing
        self.f_end = params.get("f_end", self.f_start)

        if abs(self.ramp_rate) > 1e-9:
            self._duration = abs(self.f_end - self.f_start) / abs(self.ramp_rate)
        else:
            self._duration = 0.0

        # --------------------------------------------
        # Power Factor (degrees -> radians)
        # --------------------------------------------
        
        pf_deg = master["steady_state"].get(
            "power_factor_deg",
            {"Ia": 0.0, "Ib": 0.0, "Ic": 0.0}
        )

        self.pf = {
            "Ia": pf_deg["Ia"] * pi / 180.0,
            "Ib": pf_deg["Ib"] * pi / 180.0,
            "Ic": pf_deg["Ic"] * pi / 180.0
        }

        # --------------------------------------------
        # Harmonics & OOB
        # --------------------------------------------
        
        self.harmonics = []
        self.oob = []

        # --------------------------------------------
        # Per-test amplitude scaling
        # --------------------------------------------
        
        self.voltage_pu = 1.0
        self.current_pu = 1.0

    # ==================================================
    # Phase evolution
    # ==================================================

    def theta(self, t):
        return 2 * pi * self.f_start * t + pi * self.ramp_rate * (t ** 2)

    def dtheta_dt(self, t):
        return 2 * pi * (self.f_start + self.ramp_rate * t)

    def d2theta_dt2(self, t):
        return 2 * pi * self.ramp_rate

    # ==================================================
    # Duration
    # ==================================================

    def duration(self):
        return self._duration

    # ==================================================
    # RMS per phase
    # ==================================================

    def voltage_rms_per_phase(self, t):
        Vrms = self.Vrms_nominal * self.voltage_pu
        return {
            "Va": Vrms,
            "Vb": Vrms,
            "Vc": Vrms
        }

    def current_rms_per_phase(self, t):
        Irms = self.Irms_nominal * self.current_pu
        return {
            "Ia": Irms,
            "Ib": Irms,
            "Ic": Irms
        }

    # ==================================================
    # Power factor per phase
    # ==================================================

    def power_factor_per_phase(self, t):
        return self.pf


In [10]:
    
    
class Timeline:

    def __init__(self, start_time, segments):

        self.start_time = start_time
        self.segments = segments

        self.boundaries = self._compute_boundaries()

    def _compute_boundaries(self):

        times = []
        acc = 0.0

        for seg in self.segments:
            times.append((acc, acc + seg.duration))
            acc += seg.duration

        self.total = acc
        return times

    def total_duration_sec(self):
        return self.total

    def segment_at(self, t_global):

        t_local_total = (t_global - self.start_time).total_seconds()

        for (t0, t1), seg in zip(self.boundaries, self.segments):
            if t0 <= t_local_total < t1:
                local_time = t_local_total - t0
                return seg, local_time

        raise ValueError("Time outside timeline")
    
class Segment:

    def __init__(self, state, duration, theta_offset):

        self.state = state
        self.duration = duration
        self.theta_offset = theta_offset

    def theta(self, t):
        return self.theta_offset + self.state.theta(t)

    def dtheta_dt(self, t):
        return self.state.dtheta_dt(t)

    def d2theta_dt2(self, t):
        return self.state.d2theta_dt2(t)

    def Xm(self, t):
        return self.state.Xm(t)

    def theta_end(self):
        return self.theta(self.duration)

    def contains(self, t):
        return 0 <= t < self.duration

<div style="font-size:12px">

## TimelineBuilder ##

The TimelineBuilder constructs the temporal structure of each atomic test.

It determines:

- Warm-up intervals
- Event start times
- Event durations
- Segment transitions

Each atomic test is mapped into a sequence of time-bounded segments.

The builder ensures:
- Absolute time alignment
- Deterministic event scheduling
- Phase continuity between segments

This layer translates test definitions into executable time evolution.
</div>

In [11]:
# import numpy as np
# from datetime import datetime

class TimelineBuilder:

    def __init__(self, master, atomic_test, start_time):

        self.master = master
        self.atomic = atomic_test
        self.start_time = start_time

        self.f0 = master["project"]["f0"]

    # --------------------------------------------------------
    # BUILD
    # --------------------------------------------------------
    def _steady_state(self):

        params = self.atomic.get("parameters", {})
        project = self.master["project"]

        # Use atomic frequency if provided, otherwise nominal f0
        f = params.get("frequency", project["f0"])

        return SteadyStatePhysics(
            f,
            self.master,
            self.atomic,
            nominal=False
    )
    
    def _ramp_state(self):

        params = self.atomic.get("parameters", {})
        master_meta = self.master["project"]

        ramp_rate = params.get("rate", 0.0)
        if ramp_rate == 0.0 and "ramp_rate" in params:
            ramp_rate = params["ramp_rate"]
        f_start = params["f_start"]
        return RampPhysics(
            master=self.master,
            atomic=self.atomic,
            f_start=f_start,
            ramp_rate=ramp_rate
        )    
    
    def _modulation_state(self):

        params = self.atomic.get("parameters", {})
        master_meta = self.master["project"]

        return ModulationPhysics(
            master=self.master,
            atomic=self.atomic,
            f0=master_meta["f0"],
            fm=params["fm"],
            kx=params.get("kx", 0.0),
            ka=params.get("ka", 0.0)
        )
    
        
        
    def build(self):

        segments = []

        warmup = self._build_warmup_segment()

        segments.append(warmup)

        event_segment = self._build_event_segment(
            previous_segment=warmup
        )

        segments.append(event_segment)

        return Timeline(self.start_time, segments)

    # --------------------------------------------------------
    # WARMUP
    # --------------------------------------------------------
    def _build_warmup_state(self):

        f0 = self.master["project"]["f0"]
        test_type = self.atomic["type"]
        params = self.atomic.get("parameters", {})

        if test_type in ["ramp_up", "ramp_down"]:
            f_init = params["f_start"]
        elif test_type == "steady" and "frequency" in params:
            f_init = params["frequency"]
        else:
            f_init = f0

        return SteadyStatePhysics(
            f=f_init,
            master=self.master,
            atomic=self.atomic,
            nominal=True
    )
    def _build_warmup_segment(self):

        warmup_sec = self._get_warmup_sec()

        nominal_state = self._build_warmup_state()

        return Segment(
            state=nominal_state,
            duration=warmup_sec,
            theta_offset=0.0
        )

    # --------------------------------------------------------
    # EVENT
    # --------------------------------------------------------

    def _build_event_segment(self, previous_segment):

        tag = self.atomic["type"]

        theta_end = previous_segment.theta_end()

        if tag == "steady":
            state = self._steady_state()

        elif tag == "ramp_up" or tag == "ramp_down":
            state = self._ramp_state()

        # elif tag == "mod_amp" or tag == "mod_phase":
        #     state = self._modulation_state()

        # elif tag.startswith("step"):
        #     state = self._step_state()

        else:
            raise NotImplementedError(
                f"Physics for test type '{tag}' not implemented yet."
            )

        duration = state.duration()

        return Segment(
            state=state,
            duration=duration,
            theta_offset=theta_end
        )

    def _step_state(self):

        params = self.atomic.get("parameters", {})
        master_meta = self.master["metadata"]

        return StepPhysics(
            master=self.master,
            atomic=self.atomic,
            f0=master_meta["f0"],
            kx=params.get("kx", 0.0),
            ka=params.get("ka", 0.0)
        )
    # --------------------------------------------------------
    # HELPERS
    # --------------------------------------------------------

    def _get_warmup_sec(self):

        if self.atomic["type"].startswith("steady"):
            return self.master["steady_state"]["duration_sec"]

        if self.atomic["type"].startswith("ramp"):
            return self.master["ramp_test"]["warmup_sec"]

        if self.atomic["type"].startswith("mod"):
            return self.master["modulation_test"]["warmup_sec"]

        if self.atomic["type"].startswith("step"):
            return self.master["step_test"]["warmup_sec"]

        return 1.2

<div style="font-size:12px">

### Atomic Test Expansion

`expand_atomic_tests(master)` transforms the master file into a list of atomic test cases.

Each atomic test modifies only one parameter, ensuring compliance with IEC/IEEE 60255-118-1.

The function:

- Expands steady-state ranges into discrete boundary tests
- Generates ramp up and ramp down cases
- Creates independent modulation and step tests
- Prevents parameter mixing

The resulting atomic test list is consumed by the TimelineBuilder to construct executable timelines.
</div>

In [12]:
def expand_atomic_tests(master):

    tests = []
    # helper for formatting values in tags
    def _fmt(x):
        return str(x).replace(".", "p")    
    
    # --------------------------------------------------
    # STEADY STATE
    # --------------------------------------------------

    steady = master.get("steady_state", {})
    if steady.get("enabled", False):

        # Frequency variation
        for f in steady.get("frequency_values", []):
            tests.append({
                "type": "steady",
                "tag": f"steady_freq_{_fmt(f)}Hz",
                "parameters": {
                    "frequency": f
                }
            })

        # Voltage variation
        for v in steady.get("voltage_pu_values", []):
            tests.append({
                "type": "steady",
                "tag": f"steady_volt_{_fmt(v)}pu",
                "parameters": {
                    "voltage_scale": v
                }
            })

        # Current variation
        for i in steady.get("current_pu_values", []):
            tests.append({
                "type": "steady",
                "tag": f"steady_curr_{_fmt(i)}pu",
                "parameters": {
                    "current_scale": i
                }
            })

        # Harmonics
        for h in steady.get("harmonics", []):
            tests.append({
                "type": "steady",
                "tag": f"steady_harm_{h['order']}th",
                "parameters": {
                    "harmonic": h
                }
            })

        # OOB (M class only)
        for o in steady.get("oob", []):
            tests.append({
                "type": "steady",
                "tag": f"steady_oob_{_fmt(o['frequency'])}Hz",
                "parameters": {
                    "oob": o
                }
            })

    # --------------------------------------------------
    # RAMP TEST
    # --------------------------------------------------

    ramp = master.get("ramp_test", {})
    if ramp.get("enabled", False):

        f_start, f_end = ramp["range_hz"]
        rate = ramp["ramp_rate_hz_per_s"][0]

        tests.append({
            "type": "ramp_up",
            "tag": f"ramp_up_{_fmt(f_start)}to{_fmt(f_end)}",
            "parameters": {
                "f_start": f_start,
                "f_end": f_end,
                "rate": rate
            }
        })

        tests.append({
            "type": "ramp_down",
            "tag": f"ramp_down_{_fmt(f_end)}to{_fmt(f_start)}",
            "parameters": {
                "f_start": f_end,
                "f_end": f_start,
                "rate": -rate
            }
        })

    # --------------------------------------------------
    # MODULATION TEST
    # --------------------------------------------------

    mod = master.get("modulation_test", {})
    if mod.get("enabled", False):

        # Amplitude modulation
        amp_mod = mod.get("amplitude_modulation", {})
        for fm in amp_mod.get("fm_values_hz", []):
            tests.append({
                "type": "mod_amp",
                "tag": f"mod_amp_{_fmt(fm)}Hz",
                "parameters": {
                    "kx": amp_mod["kx"],
                    "fm": fm
                }
            })

        # Phase modulation
        ph_mod = mod.get("phase_modulation", {})
        for fm in ph_mod.get("fm_values_hz", []):
            tests.append({
                "type": "mod_phase",
                "tag": f"mod_phase_{_fmt(fm)}Hz",
                "parameters": {
                    "ka": ph_mod["ka_rad"],
                    "fm": fm
                }
            })

    # --------------------------------------------------
    # STEP TEST
    # --------------------------------------------------

    step = master.get("step_test", {})
    if step.get("enabled", False):

        delta_amp = step["amplitude_step_pu"]
        delta_phi = step["phase_step_rad"]

        tests.append({
            "type": "step_amp_up",
            "tag": "step_amp_up",
            "parameters": {
                "delta_amp": delta_amp
            }
        })

        tests.append({
            "type": "step_amp_down",
            "tag": "step_amp_down",
            "parameters": {
                "delta_amp": -delta_amp
            }
        })

        tests.append({
            "type": "step_phase_adv",
            "tag": "step_phase_adv",
            "parameters": {
                "delta_phi": delta_phi
            }
        })

        tests.append({
            "type": "step_phase_ret",
            "tag": "step_phase_ret",
            "parameters": {
                "delta_phi": -delta_phi
            }
        })

    return tests

<div style="font-size:12px">

### GroundTruthGenerator

The GroundTruthGenerator evaluates the analytical model at the reporting rate (100 fps).

At each reporting instant, it computes:

- Synchrophasor phase
- Frequency
- ROCOF
- Per-phase RMS magnitudes

Frequency and ROCOF are derived analytically from θ(t), ensuring strict compliance with IEC/IEEE 60255-118-1.

Harmonics and OOB components are excluded from ground truth by design.
</div>

In [13]:
from datetime import timedelta
from numpy import angle, pi, exp, abs


class GroundTruthGenerator:

    def __init__(self, timeline, f0, report_rate):

        self.timeline = timeline
        self.f0 = f0
        self.report_rate = report_rate
        self.T = 1.0 / report_rate

    # --------------------------------------------------
    # MAIN ENTRY
    # --------------------------------------------------


    def generate(self):

        results = []

        t_global = self.timeline.start_time
        t_end = self.timeline.start_time + \
                timedelta(seconds=self.timeline.total_duration_sec())

        while t_global < t_end:

            seg, t_local = self.timeline.segment_at(t_global)

            # --------------------------------------------
            # Fundamental phase dynamics
            # --------------------------------------------

            theta = seg.theta(t_local)
            dtheta = seg.dtheta_dt(t_local)
            d2theta = seg.d2theta_dt2(t_local)

            # Instantaneous frequency
            f_inst = dtheta / (2 * pi)

            # ROCOF
            rocof = d2theta / (2 * pi)

            # Synchrophasor phase (relative to nominal)
            phi = theta - 2 * pi * self.f0 * t_local
            phi = (phi + pi) % (2 * pi) - pi

            # --------------------------------------------
            # RMS magnitudes per phase
            # --------------------------------------------

            Vrms = seg.state.voltage_rms_per_phase(t_local)
            Irms = seg.state.current_rms_per_phase(t_local)

            # Neutral current (vector sum)
            PF = seg.state.power_factor_per_phase(t_local)

            # Phasors
            Ia_vec = Irms["Ia"] * exp(1j * (-PF["Ia"]))
            Ib_vec = Irms["Ib"] * exp(1j * (-2*pi/3 - PF["Ib"]))
            Ic_vec = Irms["Ic"] * exp(1j * (2*pi/3 - PF["Ic"]))

            # Kirchhoff's current law: Ia + Ib + Ic + In = 0
            In_vec = -(Ia_vec + Ib_vec + Ic_vec)
            
            In = abs(In_vec)

            # --------------------------------------------
            # Save record
            # --------------------------------------------

            results.append({
                "absolute_time": t_global.isoformat(),

                "frequency_hz": f_inst,
                "rocof_hz_per_s": rocof,

                "phase_rad": phi,

                "voltage_rms": {
                    "Va": Vrms["Va"],
                    "Vb": Vrms["Vb"],
                    "Vc": Vrms["Vc"]
                },

                "current_rms": {
                    "Ia": Irms["Ia"],
                    "Ib": Irms["Ib"],
                    "Ic": Irms["Ic"],
                    "In": In
                }
            })

            # Advance to next report instant
            t_global += timedelta(seconds=self.T)

        return results

<div style="font-size:12px">

### WaveformGenerator

The WaveformGenerator synthesizes the seven physical channels:

- Va, Vb, Vc
- Ia, Ib, Ic
- Neutral current (In)

It includes:

- Continuous phase evolution
- Harmonic sequence enforcement (positive, negative, zero)
- Optional OOB injection
- Per-phase power factor
- Sampling drift and Gaussian jitter
- ADC-safe clipping at ±1.2 V

This module produces the physical waveform dataset used to stimulate the DUT.
</div>

In [14]:
from datetime import timedelta
from numpy import cos, pi
import random


class WaveformGenerator:

    def __init__(self, timeline, master):

        self.timeline = timeline
        self.master = master

        self.fs_nominal = master["timebase"]["sample_rate_nominal"]
        self.T_nominal = 1.0 / self.fs_nominal

        self.ppm_initial = master["timebase"]["ppm_initial"]
        self.jitter_sigma_ns = master["timebase"]["jitter_sigma_ns"]

        self.v_limit = master["scaling"]["voltage"]["max_peak"]
        self.i_limit = master["scaling"]["current"]["max_peak"]

    # ==========================================================
    # Harmonic sequence inference (physical rule)
    # ==========================================================

    def _infer_sequence(self, order):
        r = order % 3
        if r == 1:
            return "positive"
        elif r == 2:
            return "negative"
        else:
            return "zero"

    # ==========================================================
    # Hard clipping
    # ==========================================================

    def _clip(self, x, limit, channel, t_ns):

        if x > limit:
            print(f"[CLIP WARNING] {channel} exceeded +{limit:.3f} V "
                f"at t={t_ns} ns")
            return limit

        if x < -limit:
            print(f"[CLIP WARNING] {channel} exceeded -{limit:.3f} V "
                f"at t={t_ns} ns")
            return -limit

        return x
    
    # ==========================================================
    # MAIN ENTRY
    # ==========================================================

    def generate(self):

        results = []

        t_global = self.timeline.start_time
        t_end = self.timeline.start_time + \
                timedelta(seconds=self.timeline.total_duration_sec())

        t_relative = 0.0  # seconds since atomic test start

        sqrt2 = 2 ** 0.5

        while t_global < t_end:

            seg, t_local = self.timeline.segment_at(t_global)

            # --------------------------------------------------
            # Fundamental
            # --------------------------------------------------

            theta = seg.theta(t_local)

            Vrms = seg.state.voltage_rms_per_phase(t_local)
            Irms = seg.state.current_rms_per_phase(t_local)
            PF = seg.state.power_factor_per_phase(t_local)

            # Voltage fundamental
            va = sqrt2 * Vrms["Va"] * cos(theta)
            vb = sqrt2 * Vrms["Vb"] * cos(theta - 2*pi/3)
            vc = sqrt2 * Vrms["Vc"] * cos(theta + 2*pi/3)

            # Current fundamental
            ia = sqrt2 * Irms["Ia"] * cos(theta - PF["Ia"])
            ib = sqrt2 * Irms["Ib"] * cos(theta - 2*pi/3 - PF["Ib"])
            ic = sqrt2 * Irms["Ic"] * cos(theta + 2*pi/3 - PF["Ic"])

            # --------------------------------------------------
            # Harmonics (sequence-aware)
            # --------------------------------------------------

            for h in seg.state.harmonics:

                order = h["order"]
                ratio = h["ratio"]

                inferred_seq = self._infer_sequence(order)
                seq = h.get("sequence")

                if seq is None:
                    sequence = inferred_seq
                else:
                    sequence = seq
                    if sequence != inferred_seq:
                        print(f"Warning: Harmonic {order} "
                              f"sequence '{sequence}' "
                              f"≠ physical '{inferred_seq}'")

                theta_h = order * theta

                if sequence == "positive":
                    shift_a = 0.0
                    shift_b = -2*pi/3
                    shift_c = +2*pi/3

                elif sequence == "negative":
                    shift_a = 0.0
                    shift_b = +2*pi/3
                    shift_c = -2*pi/3

                elif sequence == "zero":
                    shift_a = 0.0
                    shift_b = 0.0
                    shift_c = 0.0

                else:
                    raise ValueError("Invalid harmonic sequence")

                # Voltage harmonic
                va += sqrt2 * Vrms["Va"] * ratio * cos(theta_h + shift_a)
                vb += sqrt2 * Vrms["Vb"] * ratio * cos(theta_h + shift_b)
                vc += sqrt2 * Vrms["Vc"] * ratio * cos(theta_h + shift_c)

                # Current harmonic
                ia += sqrt2 * Irms["Ia"] * ratio * \
                      cos(theta_h + shift_a - PF["Ia"])
                ib += sqrt2 * Irms["Ib"] * ratio * \
                      cos(theta_h + shift_b - PF["Ib"])
                ic += sqrt2 * Irms["Ic"] * ratio * \
                      cos(theta_h + shift_c - PF["Ic"])

            # --------------------------------------------------
            # OOB (independent frequency)
            # --------------------------------------------------

            for o in seg.state.oob:

                foob = o["frequency"]
                ratio = o["ratio"]

                theta_o = 2 * pi * foob * t_local

                va += sqrt2 * Vrms["Va"] * ratio * cos(theta_o)
                vb += sqrt2 * Vrms["Vb"] * ratio * cos(theta_o)
                vc += sqrt2 * Vrms["Vc"] * ratio * cos(theta_o)

                ia += sqrt2 * Irms["Ia"] * ratio * cos(theta_o)
                ib += sqrt2 * Irms["Ib"] * ratio * cos(theta_o)
                ic += sqrt2 * Irms["Ic"] * ratio * cos(theta_o)

            # --------------------------------------------------
            # Neutral current (physical sum)
            # --------------------------------------------------

            In = -(ia + ib + ic)

            # --------------------------------------------------
            # Clipping protection
            # --------------------------------------------------

            t_ns = int(t_relative * 1e9)

            va = self._clip(va, self.v_limit, "Va", t_ns)
            vb = self._clip(vb, self.v_limit, "Vb", t_ns)
            vc = self._clip(vc, self.v_limit, "Vc", t_ns)

            ia = self._clip(ia, self.i_limit, "Ia", t_ns)
            ib = self._clip(ib, self.i_limit, "Ib", t_ns)
            ic = self._clip(ic, self.i_limit, "Ic", t_ns)
            In = self._clip(In, self.i_limit, "In", t_ns)

            # --------------------------------------------------
            # Store sample
            # --------------------------------------------------

            results.append({
                "t_ns": t_ns,
                "samples": [
                    va, vb, vc,
                    ia, ib, ic,
                    In
                ]
            })

            # --------------------------------------------------
            # Advance time (drift + jitter)
            # --------------------------------------------------
            ppm = self.ppm_initial * 1e-6
            T_effective = self.T_nominal * (1 + ppm)

            jitter = random.gauss(
                0,
                self.jitter_sigma_ns * 1e-9
            )

            dt = T_effective + jitter

            t_relative += dt
            t_global += timedelta(seconds=dt)

        return results


<div style="font-size:12px">

### TestCampaignController

The `TestCampaignController` is the top-level orchestrator of the entire test engine.

It coordinates all layers to produce complete test artefacts for a given performance class (P or M).

Responsibilities:

- Load the master test definition
- Invoke atomic test expansion
- Assign absolute start times for each atomic test
- Ensure temporal separation between consecutive tests
- Construct timelines via `TimelineBuilder`
- Generate matched pairs of:
  - Ground truth reports
  - Waveform datasets

Each atomic test produces its own pair of output files, while absolute time continues forward across the campaign.

The controller enforces:

- Deterministic test ordering
- Non-overlapping time windows
- Consistent naming and file management
- Strict separation between specification and execution

This layer defines the overall structure of the certification campaign.

</div>

In [15]:
import json
import os
import csv
from datetime import datetime, timedelta

class TestCampaignController:

    def __init__(self,
                 master_file,
                 output_dir,
                 gap_between_tests_sec=60):

        self.master_file = master_file
        self.output_dir = output_dir
        self.gap_sec = gap_between_tests_sec

        self.master = self._load_master()
        self.atomic_tests = expand_atomic_tests(self.master)

        self.test_counter = 1

        os.makedirs(self.output_dir, exist_ok=True)

    # ----------------------------------------------------------
    # MASTER LOADING
    # ----------------------------------------------------------

    def _load_master(self):
        with open(self.master_file, "r") as f:
            return json.load(f)

    # ----------------------------------------------------------
    # MAIN ENTRY
    # ----------------------------------------------------------

    def run(self):

        base_time = datetime.fromisoformat(
            self.master["project"]["absolute_start_time"]
        )

        current_start_time = base_time

        for atomic in self.atomic_tests:

            print(f"Running test {self.test_counter}: {atomic['tag']}")

            # Build timeline
            timeline = self._build_timeline(atomic, current_start_time)

            # Generate ground truth
            gt = self._generate_ground_truth(timeline)

            # Generate waveform
            wf = self._generate_waveform(timeline)

            # Save outputs
            self._save_outputs(atomic, gt, wf)

            # Advance absolute time
            duration = timeline.total_duration_sec()
            current_start_time += timedelta(
                seconds=duration + self.gap_sec
            )

            # Force alignment to exact second boundary
            current_start_time = current_start_time.replace(
                second=0,
                microsecond=0
            )
            self.test_counter += 1

    # ----------------------------------------------------------
    # TIMELINE
    # ----------------------------------------------------------

    def _build_timeline(self, atomic_test, start_time):

        timeline = TimelineBuilder(
            master=self.master,
            atomic_test=atomic_test,
            start_time=start_time
        )

        return timeline.build()

    # ----------------------------------------------------------
    # GROUND TRUTH
    # ----------------------------------------------------------

    def _generate_ground_truth(self, timeline):

        generator = GroundTruthGenerator(
            timeline=timeline,
            f0=self.master["project"]["f0"],
            report_rate=self.master["project"]["report_rate"]
        )

        return generator.generate()

    # ----------------------------------------------------------
    # WAVEFORM
    # ----------------------------------------------------------

    def _generate_waveform(self, timeline):

        generator = WaveformGenerator(
            timeline=timeline,
            master=self.master
        )

        return generator.generate()

    # ----------------------------------------------------------
    # SAVE
    # ----------------------------------------------------------

    def _save_outputs(self, atomic_test, gt, wf):

        class_tag = self.master["project"]["class"]
        tag = atomic_test["tag"]
        test_id = f"{self.test_counter:04d}"

        base_name = f"{class_tag}_{test_id}_{tag}"

        gt_file = os.path.join(self.output_dir,
                               base_name + "_ground_truth.json")

        wf_file = os.path.join(self.output_dir,
                               base_name + "_waveform.csv")

        with open(gt_file, "w") as f:
            json.dump(gt, f, indent=2)

        with open(wf_file, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["time_ns", "Va", "Vb", "Vc", "Ia", "Ib", "Ic", "In"])
            for row in wf:
                writer.writerow([row["t_ns"]] + row["samples"])

        print(f"Saved: {base_name}")
    
    

In [16]:
# main entry point
import json

# ------------------------------------------
# Load master file
# ------------------------------------------
MASTER_FILE = "Master_M.json"
OUTPUT_DIR = "outputs_M"
# with open(MASTER_FILE, "r") as f:
#     master = json.load(f)

# ------------------------------------------
# Run campaign
# ------------------------------------------

controller = TestCampaignController(MASTER_FILE, OUTPUT_DIR)
controller.run()

Running test 1: steady_freq_45p0Hz
Saved: M_0001_steady_freq_45p0Hz
Running test 2: steady_freq_50p0Hz
Saved: M_0002_steady_freq_50p0Hz
Running test 3: steady_freq_55p0Hz
Saved: M_0003_steady_freq_55p0Hz
Running test 4: steady_volt_0p1pu
Saved: M_0004_steady_volt_0p1pu
Running test 5: steady_volt_1p0pu
Saved: M_0005_steady_volt_1p0pu
Running test 6: steady_volt_1p2pu
Saved: M_0006_steady_volt_1p2pu
Running test 7: steady_curr_0p1pu
Saved: M_0007_steady_curr_0p1pu
Running test 8: steady_curr_1p0pu
Saved: M_0008_steady_curr_1p0pu
Running test 9: steady_curr_2p0pu
Saved: M_0009_steady_curr_2p0pu
Running test 10: steady_harm_5th
Saved: M_0010_steady_harm_5th
Running test 11: steady_oob_120p0Hz
Saved: M_0011_steady_oob_120p0Hz
Running test 12: ramp_up_45p0to55p0
Saved: M_0012_ramp_up_45p0to55p0
Running test 13: ramp_down_55p0to45p0
Saved: M_0013_ramp_down_55p0to45p0
Running test 14: mod_amp_0p1Hz


NotImplementedError: Physics for test type 'mod_amp' not implemented yet.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import glob
import os

# Find CSV files
output_dir = "outputs_M"
csv_files = glob.glob(os.path.join(output_dir, "*_waveform.csv"))

if not csv_files:
    print("No CSV waveform files found in", output_dir)
else:
    # Pick the first one for verification
    csv_file = csv_files[1]
    print(f"Visualizing: {csv_file}")
    
    df = pd.read_csv(csv_file)
    
    # Create subplots: use secondary_y for current
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Add Voltage traces
    for col in ["Va", "Vb", "Vc"]:
        fig.add_trace(
            go.Scatter(x=df["time_ns"]*1e-9, y=df[col], mode="lines", name=col),
            secondary_y=False
        )
    
    # Add Current traces
    for col in ["Ia", "Ib", "Ic", "In"]:
        fig.add_trace(
            go.Scatter(x=df["time_ns"]*1e-9, y=df[col], mode="lines", name=col, line=dict(dash='dot')),
            secondary_y=True
        )
    
    fig.update_layout(
        title=f"Waveform Verification: {os.path.basename(csv_file)}",
        xaxis_title="Time (s)",
        yaxis_title="Voltage (V)",
        yaxis2_title="Current (A)",
        # template="plotly_dark"
    )
    
    fig.show()


Visualizing: outputs_M\M_0002_steady_freq_50p0Hz_waveform.csv
